# 02 - Late Delivery Label Creation

## Import Libraries and Data

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

In [ ]:
data = pd.read_csv(PROJECT_ROOT / "data" / "merged_orders.csv")

data.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_score,seller_count,seller_state_count,seller_states,seller_zip_code_prefix,customer_lat,customer_lng,seller_lat,seller_lng,distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,4.0,1.0,1.0,SP,9350.0,-23.576983,-46.587161,-23.680729,-46.444238,18.576110
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,4.0,1.0,1.0,SP,31570.0,-12.177924,-44.660711,-19.807681,-43.980427,851.495069
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,5.0,1.0,1.0,SP,14840.0,-16.745150,-48.514783,-21.363502,-48.229601,514.410666
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,5.0,1.0,1.0,MG,31842.0,-5.774190,-35.271143,-19.837682,-43.924053,1822.226336
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,5.0,1.0,1.0,SP,8752.0,-23.676370,-46.514627,-23.543395,-46.262086,29.676625


In [3]:
data.shape

(99441, 28)

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    data[col] = pd.to_datetime(data[col])

### Select Orders for Label Creation

* The prediction target is whether an order was delivered late or on time.

* To determine the true output, an order must have an actual delivery date. Therefore, only delivered orders with a non-missing `order_delivered_customer_date` are used for label creation.

* Canceled, unavailable, shipped, and other unfinished orders cannot be classified as late or on-time because their final delivery outcome is unknown.

In [5]:
labeled_data = data[
    (data["order_status"] == "delivered") &
    (data["order_delivered_customer_date"].notna())].copy()

labeled_data.shape

(96470, 28)

## Create the Late Delivery Label

An order is considered late when the actual customer delivery date is later than the estimated delivery date.

- **1 = Late**: the actual delivery date is after the estimated delivery date.
- **0 = On-time**: the actual delivery date is on or before the estimated delivery date.

In [6]:
labeled_data["is_late"] = (labeled_data["order_delivered_customer_date"] > labeled_data["order_estimated_delivery_date"]).astype(int)

labeled_data[
    ["order_delivered_customer_date",
     "order_estimated_delivery_date",
    "is_late"]].head()

,order_delivered_customer_date,order_estimated_delivery_date,is_late
0,2017-10-10 21:25:13,2017-10-18,0
1,2018-08-07 15:27:45,2018-08-13,0
2,2018-08-17 18:06:29,2018-09-04,0
3,2017-12-02 00:28:42,2017-12-15,0
4,2018-02-16 18:17:02,2018-02-26,0


**Target Distribution**

In [7]:
label_counts = labeled_data["is_late"].value_counts()
label_percentages = labeled_data["is_late"].value_counts(normalize=True) * 100


In [8]:
print("Label Counts:")
print(label_counts)

Label Counts:
is_late
0    88644
1     7826
Name: count, dtype: int64


In [9]:
print("\nLabel Percentages:")
print(label_percentages.round(2))


Label Percentages:
is_late
0    91.89
1     8.11
Name: proportion, dtype: float64


### Class Imbalance

The target variable is imbalanced:

- 91.89% of orders were delivered on time.
- 8.11% of orders were delivered late.

This class imbalance should be considered during model evaluation.

______

### Manual Label Validation

To validate the label logic, we manually inspect examples of both late and on-time orders and compare the actual delivery date with the estimated delivery date.


In [10]:
late_examples = labeled_data[
    labeled_data["is_late"] == 1][[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "is_late"
]].head(5)

late_examples

,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late
20,203096f03d82e0dffbc41ebc2e2bcfb7,2017-10-09 22:23:46,2017-09-28,1
25,fbf9ac61453ac646ce8ad9783d7d0af6,2018-03-21 22:03:54,2018-03-12,1
35,8563039e855156e48fccee4d611a3196,2018-03-20 00:59:25,2018-03-20,1
41,6ea2f835b4556291ffdc53fa0b3b95e8,2017-12-28 18:59:23,2017-12-21,1
57,66e4624ae69e7dc89bd50222b59f581f,2018-04-03 13:28:46,2018-04-02,1


In [11]:
on_time_examples = labeled_data[
    labeled_data["is_late"] == 0
][[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "is_late"
]].head(5)

on_time_examples

,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,0


In [ ]:
labeled_data.to_csv(PROJECT_ROOT / "data" / "labeled_orders.csv",index=False)

### Summary

In this notebook:

- Delivered orders with a valid actual delivery date were selected.
- The target variable `is_late` was created.
- `1` represents a late delivery, while `0` represents an on-time or early delivery.
- The target distribution was examined and showed class imbalance.
- The label logic was manually validated using examples from both classes.
- The labeled dataset was saved as `labeled_orders.csv` for the next stage of the pipeline.